In [ ]:

!pip install pathway bokeh --quiet

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from datetime import datetime
import pathway as pw
import bokeh.plotting
import panel as pn

In [ ]:

github_url = "https://raw.githubusercontent.com/Krishtiy/final-project-data-anlytics/refs/heads/main/dataset%20(6).csv"
df = pd.read_csv(github_url)

In [ ]:

grouped_datasets = {group: data for group, data in df.groupby('SystemCodeNumber')}

In [ ]:

df['SystemCodeNumber'].unique()

In [ ]:

df = grouped_datasets['BHMBCCMKT01']
df

In [ ]:

df['TrafficConditionNearby'].unique()

In [ ]:

df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)
df = df.sort_values('Timestamp').reset_index(drop=True)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

In [ ]:

df.columns = df.columns.str.strip()

df_clean = df[[
    "Timestamp", "SystemCodeNumber", "Occupancy", "Capacity",
    "QueueLength", "TrafficConditionNearby", "IsSpecialDay",
    "VehicleType", "Latitude", "Longitude"
]]

df_clean.to_csv("parking_stream.csv", index=False)

In [ ]:

class ParkingSchema(pw.Schema):
    Timestamp: str
    SystemCodeNumber: str
    Occupancy: int
    Capacity: int
    QueueLength: int
    TrafficConditionNearby: str
    IsSpecialDay: int
    VehicleType: str
    Latitude: float
    Longitude: float

In [ ]:

data = pw.demo.replay_csv("parking_stream.csv", schema=ParkingSchema, input_rate=100)

In [ ]:

fmt = "%Y-%m-%d %H:%M:%S"

data_with_time = data.with_columns(
    t=data.Timestamp.dt.strptime(fmt),
    day=data.Timestamp.dt.strptime(fmt).dt.strftime("%Y-%m-%dT00:00:00")
)

In [ ]:


import datetime

base_price = 10
alpha = 5

delta_window1 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
    )
    .with_columns(
        # price = base_price + alpha * demand_fluctuation
        # demand_fluctuation = (occ_max - occ_min) / capacity  -> normalized [0, 1]
        # Higher fluctuation = higher demand volatility = higher price
        price=base_price + alpha * (pw.this.occ_max - pw.this.occ_min) / pw.this.cap,
    )
)

In [ ]:

base_price = 10
alpha, beta, delta_coef = 0.5, 0.3, 0.4
lambda_ = 0.8


def get_traffic_weight(t: str) -> float:
    return float({"low": 0.6, "average": 1.0, "high": 1.4}.get(t, 1.0))

def get_vehicle_weight(v: str) -> float:
    return float({"car": 1.0, "bike": 0.6, "truck": 1.4, "cycle": 0.5}.get(v, 1.0))

def compute_price(
    occ_max: int,
    occ_min: int,
    cap: int,
    queue: int,
    special: int,
    vehicle: float,
    traffic: float
) -> float:
    return float(base_price) * (
        1.0 + lambda_ * (
            alpha * float(occ_max - occ_min) / float(cap)
            + beta * float(queue)
            + delta_coef * float(special)
            + vehicle
            + traffic
        ) / 5.0
    )

delta_window2 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
        IsSpecialDay=pw.reducers.max(pw.this.IsSpecialDay),
        queue=pw.reducers.max(pw.this.QueueLength),
        traffic_str=pw.reducers.any(pw.this.TrafficConditionNearby),
        vehicle_str=pw.reducers.any(pw.this.VehicleType),
    )
    .with_columns(
        # pw.apply infers return type from the function's type hints -> float
        traffic=pw.apply(get_traffic_weight, pw.this.traffic_str),
        vehicle_weight=pw.apply(get_vehicle_weight, pw.this.vehicle_str),
    )
    .with_columns(
        price=pw.apply(
            compute_price,
            pw.this.occ_max,
            pw.this.occ_min,
            pw.this.cap,
            pw.this.queue,
            pw.this.IsSpecialDay,
            pw.this.vehicle_weight,
            pw.this.traffic,
        )
    )
)

In [ ]:


base_price = 10
surge_factor = 1.5
discount_factor = 0.8
threshold_high = 0.8
threshold_low = 0.3

delta_window3 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
        IsSpecialDay=pw.reducers.max(pw.this.IsSpecialDay),
        queue=pw.reducers.max(pw.this.QueueLength),
        traffic_str=pw.reducers.any(pw.this.TrafficConditionNearby),
        vehicle_str=pw.reducers.any(pw.this.VehicleType),
    )
    .with_columns(
        occupancy_rate=(pw.this.occ_max) / pw.this.cap,
        traffic=pw.apply(
            lambda t: {"low": 0.6, "average": 1.0, "high": 1.4}.get(t, 1.0),
            pw.this.traffic_str
        ),
        vehicle_weight=pw.apply(
            lambda v: {"car": 1.0, "bike": 0.6, "truck": 1.4, "cycle": 0.5}.get(v, 1.0),
            pw.this.vehicle_str
        ),
    )
    .with_columns(

        price_multiplier=pw.apply(
            lambda rate: surge_factor if rate > threshold_high
                         else (discount_factor if rate < threshold_low else 1.0),
            pw.this.occupancy_rate
        )
    )
    .with_columns(

        price=pw.apply(
            lambda base, multiplier, special, traffic, vehicle, queue: (
                base * multiplier
                * (1.2 if special else 1.0)
                * traffic
                * vehicle
                * (1 + 0.05 * queue)
            ),
            base_price,
            pw.this.price_multiplier,
            pw.this.IsSpecialDay,
            pw.this.traffic,
            pw.this.vehicle_weight,
            pw.this.queue,
        )
    )
)

In [ ]:

pn.extension()

def price_plotter(source):
    fig = bokeh.plotting.figure(
        height=400,
        width=800,
        title="Pathway: Daily Parking Price",
        x_axis_type="datetime",
    )
    fig.line("t", "price", source=source, line_width=2, color="navy")
    # Useing scatter()
    fig.scatter("t", "price", source=source, size=6, color="red")
    return fig

In [ ]:


viz = delta_window1.plot(price_plotter, sorting_col="t")
pn.Column(viz).servable()

In [ ]:
# Cell 18 - Run Pathway pipeline
%%capture --no-display
pw.run()